# Exercise 6: Normalizing Flows
## Submission date: 30/06/2026, 23:59.

Submitted by:

 **Wael Zidan, [student ID removed]

 **Tarek Diab, [student ID removed]

<font color='red'>Before submission, please make sure that all the plots and results are present as a colab notebook file</font>


## Configurations and initializations

This section loads libraries and configurations for various tasks for this course.

We reuse the exact same setup and training loop as in exercise 5 (the VAE notebook),
and only swap the model for a normalizing flow.

In [ ]:
## Standard libraries
import os
import math
import numpy as np
import pandas as pd
from collections import defaultdict

## Imports for plotting
import matplotlib.pyplot as plt
plt.set_cmap('cividis')
%matplotlib inline
from matplotlib.colors import to_rgb
import seaborn as sns

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim

# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms

# Path to the folder where the datasets are/should be downloaded (e.g. MNIST)
DATASET_PATH = "../data"

# Path to the folder where the models are saved
CHECKPOINT_PATH = "./"

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Pick the best device available (works on Colab/CUDA, Apple MPS, or plain CPU)
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device", device)

### Continuous representation of the data

MNIST pixels are discrete (0..255). As in exercise 5 we turn them into a *continuous*
representation: we scale to $[0,1]$ and add uniform "dequantization" noise so that the
density we are learning is well defined on a continuous space. The image is then flattened
and treated as a single 784-dimensional vector.

In [ ]:
def dequantize(x):
    # x in [0,1] from ToTensor -> add uniform noise in one pixel bin, keep in [0,1)
    return (x * 255.0 + torch.rand_like(x)) / 256.0

transform = transforms.Compose([transforms.ToTensor(), dequantize])

# Loading the training dataset. We need to split it into a training and validation part
train_dataset = MNIST(root=DATASET_PATH, train=True, transform=transform, download=True)
train_set, val_set = torch.utils.data.random_split(train_dataset, [50000, 10000])

# Loading the test set
test_set = MNIST(root=DATASET_PATH, train=False, transform=transform, download=True)

# Data loaders
train_loader = data.DataLoader(train_set, batch_size=128, shuffle=True, drop_last=True, pin_memory=True, num_workers=0)
val_loader = data.DataLoader(val_set, batch_size=128, shuffle=False, drop_last=False, num_workers=0)
test_loader = data.DataLoader(test_set, batch_size=128, shuffle=False, drop_last=False, num_workers=0)

print('Train size:', len(train_loader.dataset))
print('Validation size:', len(val_loader.dataset))
print('Test size:', len(test_loader.dataset))

In [ ]:
def show_imgs(imgs):
    num_imgs = imgs.shape[0] if isinstance(imgs, torch.Tensor) else len(imgs)
    nrow = min(num_imgs, 4)
    ncol = int(math.ceil(num_imgs/nrow))
    imgs = torchvision.utils.make_grid(imgs, nrow=nrow, pad_value=128)
    imgs = imgs.clamp(min=0, max=1)
    np_imgs = imgs.cpu().numpy()
    plt.figure(figsize=(1.5*nrow, 1.5*ncol))
    plt.imshow(np.transpose(np_imgs, (1,2,0)), interpolation='nearest')
    plt.axis('off')
    plt.show()

show_imgs([train_set[i][0] for i in range(8)])

## The normalizing flow model

A normalizing flow learns an invertible map $f_\theta: x \mapsto z$ between the data and a
simple base distribution $p(z)=\mathcal{N}(0, I)$. By the change-of-variables formula

$$\log p(x) = \log p(z) + \log\left|\det \frac{\partial z}{\partial x}\right|,
\qquad z = f_\theta(x).$$

We build $f_\theta$ out of **10 affine coupling layers** (the construction from class).
Each layer splits the vector into two halves $x_1, x_2$ and maps

$$x_1' = x_1, \qquad x_2' = x_2 \odot \exp(s_\theta(x_1)) + m_\theta(x_1).$$

Because $x_1$ is left untouched, the Jacobian is triangular and its log-determinant is simply
$\sum s_\theta(x_1)$. We use a binary **checkerboard mask** to choose the split, and we
*alternate* the mask between consecutive layers so that every coordinate gets transformed.

Each of $s_\theta$ and $m_\theta$ is an MLP with **5 linear layers** and ReLU activations.
The last layer has no ReLU (it must output negative and positive values); for the scale
network $s_\theta$ we pass the last layer through a `tanh`, which keeps $\exp(s)$ in
$[e^{-1}, e^{1}]$ and makes training stable, exactly as suggested in the instructions.

Before the first coupling layer we apply a fixed `logit` squashing to map the bounded,
dequantized pixels into $\mathbb{R}$ (otherwise a Gaussian prior cannot match data living in
$[0,1]$). Its log-determinant is added to the total, so the reported log-likelihood is a
proper density.

In [ ]:
# ----- fixed logit / sigmoid squashing (with its log-determinant) -----
ALPHA = 0.05  # keep pixels away from the hard 0/1 boundaries

def logit_transform(x):
    x = ALPHA + (1 - 2 * ALPHA) * x
    y = torch.log(x) - torch.log(1 - x)
    logdet = (math.log(1 - 2 * ALPHA) - torch.log(x) - torch.log(1 - x)).sum(dim=1)
    return y, logdet

def sigmoid_transform(y):
    x = torch.sigmoid(y)
    return (x - ALPHA) / (1 - 2 * ALPHA)


class MLP(nn.Module):
    # 5 linear layers + ReLU. Last layer is linear; if bound=True a tanh squashes
    # the output to (-1, 1) -- used for the scale network s_theta.
    def __init__(self, dim, hidden=1024, bound=False):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, dim),
        )
        self.bound = bound

    def forward(self, x):
        out = self.net(x)
        return torch.tanh(out) if self.bound else out


class CouplingLayer(nn.Module):
    # Affine coupling layer with a binary mask. The masked coordinates are passed
    # through unchanged and condition the scale/translation of the rest.
    def __init__(self, dim, mask, hidden=1024):
        super().__init__()
        self.register_buffer('mask', mask)
        self.s = MLP(dim, hidden, bound=True)   # scale network  s_theta
        self.m = MLP(dim, hidden, bound=False)  # translation network  m_theta

    def forward(self, x):
        x1 = x * self.mask                       # the half that stays fixed
        s = self.s(x1) * (1 - self.mask)
        t = self.m(x1) * (1 - self.mask)
        z = x1 + (1 - self.mask) * (x * torch.exp(s) + t)
        logdet = s.sum(dim=1)
        return z, logdet

    def inverse(self, z):
        z1 = z * self.mask
        s = self.s(z1) * (1 - self.mask)
        t = self.m(z1) * (1 - self.mask)
        x = z1 + (1 - self.mask) * ((z - t) * torch.exp(-s))
        return x

In [ ]:
class NormalizingFlow(nn.Module):
    def __init__(self, dim=784, n_layers=10, hidden=1024):
        super().__init__()
        self.dim = dim
        self.logs = defaultdict(list)

        # alternating checkerboard masks (a different split in each layer)
        coords = torch.arange(dim)
        base = (coords % 2).float()
        layers = []
        for i in range(n_layers):
            mask = base if i % 2 == 0 else (1 - base)
            layers.append(CouplingLayer(dim, mask, hidden))
        self.layers = nn.ModuleList(layers)

    def log(self, key, value):
        self.logs[key].append(value)

    # ---- forward: data x -> latent z, returns z and log|det J| ----
    def forward(self, x, apply_logit=True):
        logdet = torch.zeros(x.shape[0], device=x.device)
        if apply_logit:
            x, ld = logit_transform(x)
            logdet = logdet + ld
        for layer in self.layers:
            x, ld = layer(x)
            logdet = logdet + ld
        return x, logdet

    # ---- inverse: latent z -> data x (optionally return every layer output) ----
    def inverse(self, z, to_image=True, return_intermediates=False):
        inter = [z]
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x)
            inter.append(x)
        if to_image:
            x = sigmoid_transform(x)
            inter = [sigmoid_transform(v) for v in inter]
        if return_intermediates:
            return x, inter
        return x

    # ---- negative log-likelihood used for training ----
    def calc_nll(self, x):
        z, logdet = self.forward(x)
        log_pz = (-0.5 * z ** 2 - 0.5 * math.log(2 * math.pi)).sum(dim=1)
        log_px = log_pz + logdet
        return -log_px.mean()

    # ---- draw samples from the model ----
    @torch.no_grad()
    def sample(self, num_imgs=8):
        z = torch.randn(num_imgs, self.dim, device=device)
        x = self.inverse(z)
        return x.view(num_imgs, 1, 28, 28).clamp(0, 1)

    # ----- training / evaluation steps (same interface as exercise 5) -----
    def training_step(self, batch):
        x = batch.view(batch.shape[0], -1)
        nll = self.calc_nll(x)
        self.log('train_nll', nll.item())
        return nll

    @torch.no_grad()
    def _eval_loader(self, loader):
        total, n = 0.0, 0
        for imgs, _ in tqdm(loader, leave=False):
            imgs = imgs.to(device).view(imgs.shape[0], -1)
            total += self.calc_nll(imgs).item() * imgs.shape[0]
            n += imgs.shape[0]
        return total / n

    @torch.no_grad()
    def validation_step(self, loader):
        nll = self._eval_loader(loader)
        self.log('val_nll', nll)
        return nll

    @torch.no_grad()
    def test_step(self, loader):
        nll = self._eval_loader(loader)
        self.log('test_nll', nll)
        return nll

## Model Training

In [ ]:
model = NormalizingFlow(dim=784, n_layers=10, hidden=1024)
print("Num params: {:,}".format(sum(p.numel() for p in model.parameters())))

model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.99)
epochs = 30

In [ ]:
# Training loop (same structure as exercise 5)
for epoch in range(epochs):
    model.train()
    epoch_losses = []
    for imgs, _ in tqdm(train_loader, leave=False):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        loss = model.training_step(imgs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 50.0)  # flows can spike early on
        optimizer.step()
        epoch_losses.append(loss.item())

    scheduler.step()
    val_nll = model.validation_step(val_loader)
    print(f"Epoch {epoch:2d} | train nll {np.mean(epoch_losses):8.2f} | val nll {val_nll:8.2f}")

# Test
model.eval()
final_nll = model.test_step(test_loader)
print(f"\nFinal test NLL: {final_nll:.2f} nats  ({final_nll/ (784*math.log(2)):.3f} bits/dim)")

torch.save(model.state_dict(), 'nf_model.pth')

## Question 1 - log-likelihood and training curves

We report the average log-likelihood on the test set, and plot the log-likelihood of the
train and validation sets during training. (Log-likelihood $= -\text{NLL}$, so higher is
better.)

In [ ]:
# Per-epoch training NLL (average the per-iteration values over each epoch)
iters_per_epoch = len(train_loader)
train_nll = np.array(model.logs['train_nll'])
train_nll_epoch = train_nll[:iters_per_epoch*epochs].reshape(epochs, iters_per_epoch).mean(axis=1)
val_nll_epoch = np.array(model.logs['val_nll'])

plt.figure(figsize=(7,4))
plt.plot(range(1, epochs+1), -train_nll_epoch, label='train')
plt.plot(range(1, epochs+1), -val_nll_epoch, label='validation')
plt.xlabel('epoch'); plt.ylabel('log-likelihood (nats)')
plt.title('Log-likelihood during training'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

test_nll = model.logs['test_nll'][-1]
print(f"Test log-likelihood: {-test_nll:.2f} nats")
print(f"Test NLL: {test_nll:.2f} nats  ({test_nll/(784*math.log(2)):.3f} bits/dim)")

### Generated samples

In [ ]:
model.eval()
samples = model.sample(num_imgs=16)
show_imgs(samples)

## Question 2 - from latent to data, layer by layer

For 7 latent samples $z$ we show 11 images: the (reshaped) latent sample, then the output
after inverting each of the 10 coupling layers, ending at the final image $x$. Because the
coupling layers act on the logit-space vector, we map every intermediate back through the
sigmoid so it can be displayed as an image. The grid has 7 rows and 11 columns.

In [ ]:
model.eval()
n_rows = 7
with torch.no_grad():
    z = torch.randn(n_rows, model.dim, device=device)
    _, inter = model.inverse(z, return_intermediates=True)   # 11 tensors: z + 10 layer outputs

# inter[k] has shape (7, 784); build the 7 x 11 grid (z + 10 layer outputs)
n_cols = len(inter)                              # == 11 for the 10-layer model
grid = torch.stack(inter, dim=1)                 # (7, n_cols, 784)
grid = grid.reshape(n_rows*n_cols, 1, 28, 28).clamp(0, 1).cpu()

plt.figure(figsize=(n_cols*1.1, n_rows*1.1))
g = torchvision.utils.make_grid(grid, nrow=n_cols, pad_value=0.5)
plt.imshow(np.transpose(g.numpy(), (1,2,0)), interpolation='nearest')
plt.axis('off')
plt.title('latent  z  →  output of each NF layer  →  sample  x', fontsize=10)
plt.show()

## Question 3 - latent traversals

We pick two images of different digits, encode each to its latent representation
$z = f_\theta(x)$, and walk along the straight line between $z_1$ and $z_2$ in 10 steps.
Each point is decoded back with the inverse map. We repeat this for 3 different pairs and
should see the first digit morph into the second.

In [ ]:
def get_image_by_label(dataset, label):
    for img, y in dataset:
        if y == label:
            return img
    raise ValueError(label)

pairs = [(0, 1), (3, 8), (4, 9)]
n_steps = 10
model.eval()

plt.figure(figsize=(n_steps*1.1, len(pairs)*1.1))
for r, (d1, d2) in enumerate(pairs):
    x1 = get_image_by_label(test_set, d1).view(1, -1).to(device)
    x2 = get_image_by_label(test_set, d2).view(1, -1).to(device)
    with torch.no_grad():
        z1, _ = model.forward(x1)
        z2, _ = model.forward(x2)
        alphas = torch.linspace(0, 1, n_steps, device=device).view(-1, 1)
        zs = (1 - alphas) * z1 + alphas * z2      # linear path in latent space
        xs = model.inverse(zs).view(n_steps, 1, 28, 28).clamp(0, 1).cpu()
    for c in range(n_steps):
        ax = plt.subplot(len(pairs), n_steps, r*n_steps + c + 1)
        ax.imshow(xs[c, 0].numpy(), cmap='gray'); ax.axis('off')
plt.suptitle('Latent traversals between pairs of digits', y=1.02)
plt.tight_layout()
plt.show()

## Summary

To wrap things up: we built a RealNVP-style normalizing flow on MNIST out of 10 affine
coupling layers. In each layer we keep one half of the flattened image fixed and use it to
predict a scale $s_\theta$ and a shift $m_\theta$ for the other half. The alternating
checkerboard mask is what makes sure every coordinate eventually gets transformed. Both
$s_\theta$ and $m_\theta$ are 5-layer ReLU MLPs, and we put a `tanh` on the scale so that
$\exp(s)$ stays bounded and the training doesn't blow up early on.

The nice property of coupling layers is that the Jacobian is triangular, so its log-determinant
is just $\sum s_\theta$. That lets us train by directly maximizing the exact log-likelihood
under a standard Gaussian prior (we also add the fixed logit transform first, otherwise pixels
living in $[0,1]$ can't really match a Gaussian).

After about 30 epochs the train/validation log-likelihood curves flatten out and most of the
samples look like actual digits. The layer-by-layer plot in Q2 shows the noise slowly turning
into a digit across the 10 layers, and the Q3 traversals morph one digit into another fairly
smoothly, so the latent space we learned seems to be well behaved.